In [ ]:
%pip install seaborn
%pip install sentence_transformers

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
from sentence_transformers import SentenceTransformer, util

In [ ]:

env_path = Path('../../.env')
load_dotenv(dotenv_path=env_path)

In [ ]:

print("Carregando modelo de Embeddings")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
caminhos = {
    "Corretor": "../../../BASES/experimento_corretor/analise_entropia/corretor_dados_anonimizados.csv",
    "BERTimbau": "../../../BASES/experimento_corretor/analise_entropia/bertimbau_entropy.csv",
    "spaCy": "../../../BASES/experimento_corretor/analise_entropia/spacy_entropy.csv"
}

In [ ]:
resultados = []

In [ ]:
for modelo_nome, caminho in caminhos.items():
    print(f"\n--- Processando modelo: {modelo_nome} ---")
    
    df = pd.read_csv(caminho)
    
    df['relato'] = df['relato'].astype(str)
    df['relato_anon'] = df['relato_anon'].astype(str)
    
    textos_orig = df['relato'].tolist()
    textos_anon = df['relato_anon'].tolist()
    
    print("Gerando embeddings dos relatos originais...")
    emb_orig = model.encode(textos_orig, convert_to_tensor=True, show_progress_bar=True)
    
    print("Gerando embeddings dos relatos anonimizados...")
    emb_anon = model.encode(textos_anon, convert_to_tensor=True, show_progress_bar=True)
    
    print("Calculando Similaridade por Cosseno...")
    cosseno_scores = util.cos_sim(emb_orig, emb_anon)
    
    similaridades = np.diag(cosseno_scores.cpu().numpy())
    
    media = similaridades.mean()
    std = similaridades.std()
    
    resultados.append({
        "Modelo": modelo_nome,
        "Similaridade_Cosseno_Media": media,
        "Similaridade_Percentual": media * 100,
        "Desvio_Padrao": std
    })

In [ ]:
df_resultados = pd.DataFrame(resultados)
print("\n" + "="*50)
print("🏆 RESULTADO FINAL - RETENÇÃO DE UTILIDADE (SEMÂNTICA) 🏆")
print("="*50)
print(df_resultados[['Modelo', 'Similaridade_Percentual', 'Desvio_Padrao']])
print("="*50)

In [ ]:
df_resultados.to_csv("../../../BASES/experimento_corretor/avaliacao_modelos/comparativo_cosseno_final.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Modelo', y='Similaridade_Percentual', data=df_resultados, palette='viridis')

for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 10), 
                textcoords='offset points', fontsize=12, fontweight='bold')

plt.ylim(0, 100)
plt.title('Comparativo de Retenção Semântica após Anonimização (Similaridade por Cosseno)', fontsize=14, pad=15)
plt.ylabel('Similaridade Retida (%)', fontsize=12)
plt.xlabel('Modelo de Extração (NER)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)   

In [ ]:
plt.savefig("../../images/grafico_similaridade_cosseno.png", bbox_inches='tight', dpi=300)
plt.show()

print("\nGráfico exportado com sucesso para a pasta 'images'!")